In [9]:
# Setup: paths and dependency check
import os, sys, subprocess
import pandas as pd

BASE_DIR = os.getcwd()
JSON_DIR = os.path.join(BASE_DIR, "Json")
print("BASE_DIR:", BASE_DIR)
print("JSON_DIR:", JSON_DIR)

# Ensure langdetect is available for EnglishFilter
try:
    import langdetect  # noqa: F401
    print("OK: langdetect available")
except ImportError:
    print("Installing langdetect...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langdetect"])
    import langdetect  # noqa: F401
    print("Installed langdetect")

BASE_DIR: k:\GithubRepo\comment-classification\src\Preprocess2
JSON_DIR: k:\GithubRepo\comment-classification\src\Preprocess2\Json
OK: langdetect available


In [29]:
# Test: EmojiDecoder
from Decoders.demoji import EmojiDecoder
import os

decoder = EmojiDecoder(os.path.join(JSON_DIR, "emoji_vi.json"))
s = "Haha 😂❤️ ok"
decoded = decoder.decode(s)
assert ":cười_ra_nước_mắt:" in decoded
assert ":trái_tim_đỏ:" in decoded
removed = decoder.remove_emoji(s)
assert "😂" not in removed and "❤️" not in removed
assert decoder.count_emojis(s) == 2
print("EmojiDecoder OK")
print("Decoded:", decoded)
print("Removed:", removed)

EmojiDecoder OK
Decoded: Haha :cười_ra_nước_mắt::trái_tim_đỏ: ok
Removed: Haha  ok


In [30]:
# Test: TeencodeConverter
from Decoders.teencode_decoder import TeencodeConverter
import os

tc = TeencodeConverter(os.path.join(JSON_DIR, "teencode.json"))
text = "Tui khum thich fb, hnay đi sg nha"
out = tc.replace(text)
print("Input:", text)
print("Output:", out)
assert "Tôi" in out and "không" in out and "thích" in out and "facebook" in out
print("TeencodeConverter OK")

Input: Tui khum thich fb, hnay đi sg nha
Output: Tôi không thích facebook, hôm nay đi sài gòn nhé
TeencodeConverter OK


In [21]:
# Test: RepetitionDecoder
from Decoders.repetition_decoder import RepetitionDecoder

rd = RepetitionDecoder()
inp = "tôiiiiiiiiii mẹeeeeeee!!!! :))))"
out = rd.normalize(inp)
print("Input:", inp)
print("Output:", out)
assert "tôi" in out and "mẹ" in out and "!!!!" in out and ":))))" in out
print("RepetitionDecoder OK")

Input: tôiiiiiiiiii mẹeeeeeee!!!! :))))
Output: tôi mẹe!!!! :))))
RepetitionDecoder OK


In [22]:
# Test: IconNormalizer
from Normalizers.icon_normalizer import IconNormalizer
import os

inorm = IconNormalizer(os.path.join(JSON_DIR, "icons.json"))
s = ":)))) =)))) <33333 :((("
out = inorm.normalize(s)
print("Input:", s)
print("Output:", out)
assert ":))" in out and "=))" in out and "<333" in out and ":((" in out
print("IconNormalizer OK")

Input: :)))) =)))) <33333 :(((
Output: :)) =)) <333 :((
IconNormalizer OK


In [24]:
# Test: VietnameseTypingNormalizer
from Normalizers.vietnamese_typing_normalizer import VietnameseTypingNormalizer

vt = VietnameseTypingNormalizer()
s = "oà uý Oá Uỵ oé"
out = vt.replace(s)
print("Input:", s)
print("Output:", out)
assert "òa" in out and "úy" in out and "Óa" in out and "Ụy" in out and "óe" in out
print("VietnameseTypingNormalizer OK")

Input: oà uý Oá Uỵ oé
Output: òa úy Óa Ụy óe
VietnameseTypingNormalizer OK


In [25]:
# Test: EnglishFilter
from Filters.english_filter import EnglishFilter
import pandas as pd

ef = EnglishFilter()
df = pd.DataFrame({"text": ["Tôi đang học máy", "This is a test", "Xin chào các bạn"]})
res = ef.filt(df, "text")
print(res)
assert res["is_vi"].all()
assert len(res) == 2
print("EnglishFilter OK")

Còn lại 2/3 sau EnglishFilter
               text  vi_chance  is_vi
0  Tôi đang học máy   0.999997   True
2  Xin chào các bạn   0.999999   True
EnglishFilter OK


In [26]:
# Test: LinkFilter
from Filters.link_filter import LinkFilter
import pandas as pd

df_links = pd.DataFrame({
    "text": [
        "Xem video đi",
        "http://example.com",
        "www.youtube.com/watch?v=abc",
        "my-domain.io/path"
    ]
})
lf = LinkFilter()
out_links = lf.filt(df_links, "text")
print(out_links)
assert out_links["text"].tolist() == ["Xem video đi"]
print("LinkFilter OK")

Còn lại 1/4 sau LinkFilter
           text
0  Xem video đi
LinkFilter OK


k:\GithubRepo\comment-classification\src\Preprocess2\Filters\link_filter.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = ~df[text_col].astype(str).str.contains(self.url_pattern)


In [27]:
# Test: ShortTextFilter
from Filters.short_text_filter import ShortTextFilter
import pandas as pd

df_short = pd.DataFrame({
    "text": [
        "Hi",
        "Xin chào",
        "Hôm nay trời đẹp và nhiều mây"
    ]
})
sf = ShortTextFilter()
out_short = sf.filt(df_short, "text", len_threshold=5, space_threshold=2)
print(out_short)
assert out_short["text"].tolist() == ["Xin chào", "Hôm nay trời đẹp và nhiều mây"]
print("ShortTextFilter OK")

Còn lại 2/3 sau ShortTextFilter
                            text
1                       Xin chào
2  Hôm nay trời đẹp và nhiều mây
ShortTextFilter OK


In [28]:
# Optional Test: VnCoreNLPDecoder (guarded)
from Decoders.vncorenlp_decoder import VnCoreNLPDecoder
import os

MODEL_DIR = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "vncorenlp")
ICON_JSON = os.path.join(JSON_DIR, "icons.json")
JAVA_CANDIDATES = [r"C:\\Program Files\\Java\\jdk-21", os.environ.get("JAVA_HOME")]
JAVA_HOME = next((p for p in JAVA_CANDIDATES if p and os.path.exists(p)), None)

print("MODEL_DIR:", MODEL_DIR)
print("JAVA_HOME:", JAVA_HOME)

if os.path.exists(MODEL_DIR) and JAVA_HOME:
    try:
        dec = VnCoreNLPDecoder(MODEL_DIR, ICON_JSON, java_home=JAVA_HOME)
        sample = "Son_Tung M-TP :)))) ơi!!"
        out = dec.segment_text(sample)
        print("Segmented:", out)
        assert "M-TP" in out
        print("VnCoreNLPDecoder OK")
    except AttributeError as e:
        print("VnCoreNLPDecoder missing attribute:", e)
    except Exception as e:
        print("VnCoreNLPDecoder skipped due to error:", e)
else:
    print("Skipping VnCoreNLPDecoder test: Java or models missing.")

MODEL_DIR: k:\GithubRepo\comment-classification\vncorenlp
JAVA_HOME: C:\\Program Files\\Java\\jdk-21
Skipping VnCoreNLPDecoder test: Java or models missing.
